<img src="../../../docs/assets/0.-BC-dev-hub-LOGO-flicker.svg" alt="BrainChip Dev Hub" width="200"/>

# Visual Wake Words on Akida 1

<p align="right">
Run Time: ~1 hour with training included / ~2 minutes with training skipped
</p>

This notebook steps through the full pipeline for a **Visual Wake Words (VWW)**
binary classifier on **Akida 1 (AKD1500)**: training a tf_keras model,
quantization and conversion to Akida format, and evaluation of the resulting
Akida model. For background on the VWW task, dataset, and model performance
benchmarks, see the [README](README.md).

The focus is on the **Akida-specific** aspects of the pipeline. The full data
preprocessing and training code is available in the accompanying Python files —
it is standard tf_keras code and is not described further in this notebook.

By default, model training is run to ensure reproducibility. However, you can
cut the running time of the notebook to under a minute if desired by 
skipping the training runs and downloading pretrained float and quantized models
instead: simply set the relevant `RUN_FLOAT_TRAINING` and `RUN_QAT_TRAINING`
variables in the first code cell to `False`.

## Setup

The default dataset path is `./data/vw_coco2014_96`. See the [README](README.md)
for download instructions and symlink setup. Update `DATA_PATH` below if needed.

In [2]:
import os
import numpy as np
import tensorflow as tf
from tqdm import tqdm

DATA_PATH = './data/vw_coco2014_96'
MODELS_DIR = './models'
os.makedirs(MODELS_DIR, exist_ok=True)

RUN_FLOAT_TRAINING = True
RUN_QAT_TRAINING = True

SEED = 42

# Must be called before any TF ops to make GPU ops (conv backward passes,
# bilinear resize, etc.) deterministic. Has a small throughput cost.
tf.config.experimental.enable_op_determinism()

## Dataset

`get_data` returns a training and a validation `tf.data.Dataset`. The full
preprocessing and augmentation code is in [vww_data.py](vww_data.py) —
standard tf_keras pipeline code, not described further here.

In [3]:
from vww_data import get_data

BATCH_SIZE = 32
INPUT_SHAPE = (96, 96, 3)

train_ds, val_ds = get_data(DATA_PATH, INPUT_SHAPE, BATCH_SIZE, seed=SEED)

Found 98658 images belonging to 2 classes.
Found 10961 images belonging to 2 classes.


## Model

The backbone is **AkidaNet** — a MobileNet V1 variant whose layer structure
maps efficiently onto Akida hardware — with width multiplier `alpha=0.25`.
The narrow variant keeps parameter count low while retaining sufficient
capacity for the binary VWW task.

Key design choices:

- **Input resolution 96×96**, down from the 224×224 used for the ImageNet
  base model.
- **2-class output head** (person / non-person), returning raw logits — no
  softmax. The loss function handles the softmax implicitly via
  `from_logits=True`.
- **`input_scaling=(255, 0)`** embeds a linear mapping from uint8 [0, 255]
  to float [0, 1] as the first layer. This keeps the data pipeline
  hardware-friendly — inputs never need to be normalised outside the model.
- **`AkidaVersion.v1` context** constrains the layer configuration to what
  the AKD1500 hardware supports.

In [4]:
from vww_model import build_vww_model
model = build_vww_model(seed=SEED)


I0000 00:00:1783890897.543076  201207 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22284 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:04:00.0, compute capability: 8.6
/home/dmclelland/miniconda3/envs/brainchip_devhub_env/lib/python3.12/site-packages/akida_models/model_io.py:147: UserWarning: Model akidanet_imagenet_224_alpha_25.h5 has been trained with akida_models 1.1.10 which is the last version supporting 1.0 models training. Continuing execution.
  warnings.warn(f'Model {model_name_v1} has been trained with akida_models 1.1.10 which is '


## Float Training

The model is trained for 50 epochs using Adam with a step-decay learning rate
schedule. The full training code is in [vww_train.py](vww_train.py) —
standard tf_keras code, not described further here.


In [5]:
from vww_train import train_vww


if RUN_FLOAT_TRAINING:
    LEARNING_RATE = 1e-3
    EPOCHS = 20
    # Make sure the dataset seed is freshly set for reproducibility
    train_ds, val_ds = get_data(DATA_PATH, INPUT_SHAPE, BATCH_SIZE, seed=SEED)

    train_vww(model, train_ds, val_ds,
            EPOCHS,
            LEARNING_RATE,
            seed=SEED
            )

    float_model_path = os.path.join(MODELS_DIR, 'akidanet_vww.h5')
    model.save(float_model_path, include_optimizer=False)
    print(f'Float model saved to {float_model_path}')
else:
    from tf_keras.models import load_model
    print('Training skipped. Using pretrained model...')
    model_path = 'pretrained_models/akidanet_vww.h5'
    model = load_model(model_path)

Found 98658 images belonging to 2 classes.
Found 10961 images belonging to 2 classes.
Epoch 1/20


I0000 00:00:1783890900.794973  201359 cuda_dnn.cc:529] Loaded cuDNN version 90300


3084/3084 [==============================] - 127s 41ms/step - loss: 0.6449 - accuracy: 0.7737 - val_loss: 0.5579 - val_accuracy: 0.8083
Epoch 2/20
3084/3084 [==============================] - 125s 41ms/step - loss: 0.4864 - accuracy: 0.8346 - val_loss: 0.5962 - val_accuracy: 0.7758
Epoch 3/20
3084/3084 [==============================] - 126s 41ms/step - loss: 0.4724 - accuracy: 0.8388 - val_loss: 0.4870 - val_accuracy: 0.8257
Epoch 4/20
3084/3084 [==============================] - 128s 41ms/step - loss: 0.4510 - accuracy: 0.8463 - val_loss: 0.6158 - val_accuracy: 0.7837
Epoch 5/20
3084/3084 [==============================] - 125s 41ms/step - loss: 0.4345 - accuracy: 0.8505 - val_loss: 0.5322 - val_accuracy: 0.8033
Epoch 6/20
3084/3084 [==============================] - 126s 41ms/step - loss: 0.4174 - accuracy: 0.8564 - val_loss: 0.7843 - val_accuracy: 0.7158
Epoch 7/20
3084/3084 [==============================] - 125s 41ms/step - loss: 0.4134 - accuracy: 0.8564 - val_loss: 0.8596 - val

/home/dmclelland/miniconda3/envs/brainchip_devhub_env/lib/python3.12/site-packages/tf_keras/src/engine/training.py:3098: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


### Evaluate float model

In [6]:
model.compile(metrics=['accuracy'])
_, float_accuracy = model.evaluate(val_ds, verbose=0)
print(f'Float validation accuracy: {float_accuracy:.4f}')

Float validation accuracy: 0.8933


## Quantization

Akida 1 operates with integer weights and activations. We use `cnn2snn` to
quantize the float model to 4 bits for both weights and activations (8-bit
weights are enabled for the first layer only, which is also unusual in
receiving uint8 inputs):

Post-training quantization maps the float parameters to their nearest
representable integer values. Some accuracy is typically lost in this step,
which the subsequent QAT pass recovers.

Note: the quantized model can be saved using the standard method. However,
for later reloading, because of the custom quantized layers in the model
we have to use the `load_quantized_model` function from cnn2snn (a wrapper
around the standard tf_keras loading function)

In [7]:
from cnn2snn import quantize, load_quantized_model

quantized_model = quantize(
    model,
    input_weight_quantization=8,
    weight_quantization=4,
    activ_quantization=4,
)
ptq_model_path = os.path.join(MODELS_DIR, 'akidanet_vww_ptq.h5')
quantized_model.save(ptq_model_path, include_optimizer=False)

del quantized_model

quantized_model = load_quantized_model(ptq_model_path)

### Quantization-Aware Training (QAT)

A few epochs of fine-tuning sufficient to recover most of the accuracy lost 
during quantization. It is typical to find that a lower learning rate (e.g. /10)
is required during this phase than during the initial training.

Note that, although Quatization Aware Training can sound intimidating,
the model quantized via `cnn2snn` can simply be reinserted into the 
same training function that was used for the initial float training.

In [8]:

if RUN_QAT_TRAINING:
    QAT_LEARNING_RATE = 1e-4
    QAT_EPOCHS = 5
    
    # We re-fetch the dataset just for reproducibility vs the script version of the code
    train_ds, val_ds = get_data(DATA_PATH, INPUT_SHAPE, BATCH_SIZE, seed=SEED)
    
    train_vww(quantized_model, train_ds, val_ds,
            QAT_EPOCHS,
            QAT_LEARNING_RATE,
            )

    qat_model_path = os.path.join(MODELS_DIR, 'akidanet_vww_qat.h5')
    quantized_model.save(qat_model_path, include_optimizer=False)
    print(f'QAT model saved to {qat_model_path}')
else:
    print('QAT skipped. Loading pretrained model...')
    q_model_path = 'pretrained_models/akidanet_vww_qat.h5'
    quantized_model = load_quantized_model(q_model_path)

Found 98658 images belonging to 2 classes.
Found 10961 images belonging to 2 classes.
Epoch 1/5
3084/3084 [==============================] - 143s 45ms/step - loss: 0.3430 - accuracy: 0.8611 - val_loss: 0.3320 - val_accuracy: 0.8697
Epoch 2/5
3084/3084 [==============================] - 139s 45ms/step - loss: 0.3256 - accuracy: 0.8696 - val_loss: 0.3347 - val_accuracy: 0.8667
Epoch 3/5
3084/3084 [==============================] - 136s 44ms/step - loss: 0.3076 - accuracy: 0.8783 - val_loss: 0.3273 - val_accuracy: 0.8727
Epoch 4/5
3084/3084 [==============================] - 136s 44ms/step - loss: 0.2863 - accuracy: 0.8891 - val_loss: 0.3082 - val_accuracy: 0.8813
Epoch 5/5
3084/3084 [==============================] - 138s 45ms/step - loss: 0.2645 - accuracy: 0.8998 - val_loss: 0.3035 - val_accuracy: 0.8854
QAT model saved to ./models/akidanet_vww_qat.h5


### Evaluate quantized model

In [9]:
quantized_model.compile(metrics=['accuracy'])
_, qat_accuracy = quantized_model.evaluate(val_ds, verbose=0)
print(f'QAT validation accuracy: {qat_accuracy:.4f}')

QAT validation accuracy: 0.8854


## Conversion to Akida Format

`cnn2snn.convert` compiles the quantized Keras model into an Akida `.fbz`
model that can be loaded and executed directly on AKD1500 hardware.
The converter verifies hardware compatibility and maps each layer to its
corresponding Akida primitive.

In [10]:
from cnn2snn import convert

akida_model = convert(quantized_model)

akida_model_path = os.path.join(MODELS_DIR, 'akidanet_vww_qat.fbz')
akida_model.save(akida_model_path)
print(f'Akida model saved to {akida_model_path}')
akida_model.summary()

Akida model saved to ./models/akidanet_vww_qat.fbz
                Model Summary                 
______________________________________________
Input shape  Output shape  Sequences  Layers
[96, 96, 3]  [1, 1, 2]     1          15    
______________________________________________

__________________________________________________________
Layer (type)              Output shape  Kernel shape    

============ SW/conv_0-predictions (Software) ============

conv_0 (InputConv.)       [48, 48, 8]   (3, 3, 3, 8)    
__________________________________________________________
conv_1 (Conv.)            [48, 48, 16]  (3, 3, 8, 16)   
__________________________________________________________
conv_2 (Conv.)            [24, 24, 32]  (3, 3, 16, 32)  
__________________________________________________________
conv_3 (Conv.)            [24, 24, 32]  (3, 3, 32, 32)  
__________________________________________________________
separable_4 (Sep.Conv.)   [12, 12, 64]  (3, 3, 32, 1)   
___________________

## Evaluation of Akida Model

We now run evaluation through the Akida model, to check that accuracy is 
comparable to that obtained from the quantized tf_keras model. He, we deliberately
use the software backend (the default, since we do not check for and map to
a connected hardware device): this delivers a
bit-accurate simulation of the results that will be obtained when running
the model on hardware.

In the accompanying [vww_notebook_benchmark.ipynb](plant_village_notebook_benchmark.ipynb)
the same evaluation is run using the hardware backend (if, of course, a hardware Akida
device is connected), allowing you to confirm that the results are identical.

### Run Evaluation on Akida

The Akida runtime cannot consume `tf.data.Dataset` objects directly, rather
it expects a 4D numpy array (n, h, w, c) in uint8 format. So we
iterate over validation batches manually.

The model output tensor has shape `(B, 1, 1, C)` which is squeezed to 
`(B, C)` before taking the class argmax.

In [11]:

labels_all = []
logits_all = []
num_batches = len(val_ds)
val_ds.reset() # Ensure a clean run through the full validation dataset
for _ in tqdm(range(num_batches), desc="Evaluating on Akida"):
    batch, label_batch = next(val_ds)
    if not isinstance(batch, np.ndarray):
        batch = batch.numpy()

    logits_batch = akida_model.predict(batch.astype(np.uint8))

    logits_batch = logits_batch.squeeze(axis=(1, 2))
    labels_all.append(label_batch)
    logits_all.append(logits_batch)

labels_all = np.concatenate(labels_all)
logits_all = np.concatenate(logits_all)
preds = np.argmax(logits_all, axis=1)

akida_accuracy = float(np.mean(preds == np.array(labels_all)))
print(f'Akida accuracy: {akida_accuracy:.4f}')

Evaluating on Akida: 100%|██████████| 343/343 [00:07<00:00, 42.91it/s]

Akida accuracy: 0.8842


### Activation Sparsity

Akida hardware skips computation for zero-valued activations, so activation
sparsity directly reduces both energy consumption and inference latency.
Below we measure per-layer sparsity on a 1024-sample calibration batch drawn
from the training set.

In [12]:
from akida_models.sparsity import compute_sparsity
from brainchip_utils.plot_utils import pretty_print_sparsity
from vww_data import get_samples

NUM_SAMPLES=1024
samples = get_samples(DATA_PATH, INPUT_SHAPE, num_samples=NUM_SAMPLES)
sparsity_dict = compute_sparsity(akida_model, samples=samples)
pretty_print_sparsity(sparsity_dict)

Found 109619 images belonging to 2 classes.

Layer            Sparsity
-------------------------
conv_0            23.86%
conv_1            37.03%
conv_2            56.00%
conv_3            40.28%
separable_4       33.30%
separable_5       65.21%
separable_6       40.46%
separable_7       48.44%
separable_8       60.60%
separable_9       65.68%
separable_10      74.43%
separable_11      59.27%
separable_12      89.35%
separable_13      89.25%
predictions        0.50%
-------------------------
Mean              52.24%


## Summary


In [13]:
print(f"Float: {float_accuracy:.4f}")
print(f"QAT:   {qat_accuracy:.4f}")
print(f"Akida: {akida_accuracy:.4f}")

Float: 0.8933
QAT:   0.8854
Akida: 0.8842
